In [0]:
%run ./02_silver_logic

In [0]:
from delta.tables import DeltaTable

full = build_silver(spark)
max_date = full.agg(F.max("date")).first()[0]
new_batch = full.filter(F.col("date") >= F.date_sub(F.lit(max_date), 27))

before = spark.table("workspace.m5.silver_sales_daily").count()

(DeltaTable.forName(spark, "workspace.m5.silver_sales_daily").alias("t")
    .merge(new_batch.alias("s"), "t.id = s.id AND t.date = s.date")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

after = spark.table("workspace.m5.silver_sales_daily").count()
print(f"before: {before:,}  after: {after:,}  added: {after - before:,}")

In [0]:
display(spark.sql("DESCRIBE HISTORY workspace.m5.silver_sales_daily"))
print("rows in version 0:", spark.sql("SELECT COUNT(*) FROM workspace.m5.silver_sales_daily VERSION AS OF 0").first()[0])